In [54]:
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import mm
from reportlab.lib import colors
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer, Image as RLImage
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER, TA_LEFT, TA_JUSTIFY
from datetime import datetime

def generate_dr_report(patient_data, gradcam_path, shap_path, output_path):
    doc = SimpleDocTemplate(output_path, pagesize=A4,
                             topMargin=15*mm, bottomMargin=15*mm,
                             leftMargin18=18*mm, rightMargin=18*mm)

    styles = getSampleStyleSheet()

    title_style = ParagraphStyle('TitleStyle', parent=styles['Heading1'],
                                   fontSize=16, textColor=colors.HexColor('#1F4E79'), spaceAfter=4)
    subtitle_style = ParagraphStyle('SubtitleStyle', parent=styles['Normal'],
                                      fontSize=9, textColor=colors.HexColor('#666666'), spaceAfter=12)
    section_style = ParagraphStyle('SectionStyle', parent=styles['Heading2'],
                                     fontSize=11, textColor=colors.HexColor('#2E75B6'),
                                     spaceBefore=12, spaceAfter=6)
    body_style = ParagraphStyle('BodyStyle', parent=styles['Normal'], fontSize=9, leading=13)
    disclaimer_style = ParagraphStyle('DisclaimerStyle', parent=styles['Normal'],
                                        fontSize=7, textColor=colors.HexColor('#888888'),
                                        fontName='Helvetica-Oblique')

    elements = []

    # ── HEADER ──
    elements.append(Paragraph("Diabetic Retinopathy Screening Report", title_style))
    elements.append(Paragraph("AI-assisted clinical decision support — for physician review only", subtitle_style))

    report_id = f"DR-{datetime.now().strftime('%Y%m%d')}-{np.random.randint(1000,9999)}"
    elements.append(Paragraph(f"<b>Report ID:</b> {report_id} &nbsp;&nbsp; <b>Generated:</b> {datetime.now().strftime('%d %B %Y, %H:%M')}", body_style))
    elements.append(Spacer(1, 10))

    # ── RISK BANNER ──
    risk_score = patient_data['clinical_risk_score']
    risk_level = "LOW" if risk_score < 30 else "MODERATE" if risk_score < 60 else "HIGH"
    risk_color = colors.HexColor('#27A745') if risk_level == "LOW" else colors.HexColor('#EF9F27') if risk_level == "MODERATE" else colors.HexColor('#E24B4A')

    risk_table = Table([
        [f"{risk_score}%", f"{risk_level} RISK\nPredicted: {patient_data['predicted_stage_label']} (Stage {patient_data['predicted_dr_stage']})\nImage model confidence: {patient_data['image_confidence']}%"]
    ], colWidths=[40*mm, 130*mm])
    risk_table.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,-1), colors.HexColor('#FCEBEB') if risk_level=="HIGH" else colors.HexColor('#FFF8E1') if risk_level=="MODERATE" else colors.HexColor('#E8F5E9')),
        ('TEXTCOLOR', (0,0), (0,0), risk_color),
        ('FONTSIZE', (0,0), (0,0), 24),
        ('FONTNAME', (0,0), (0,0), 'Helvetica-Bold'),
        ('FONTSIZE', (1,0), (1,0), 9),
        ('VALIGN', (0,0), (-1,-1), 'MIDDLE'),
        ('LEFTPADDING', (0,0), (-1,-1), 12),
        ('TOPPADDING', (0,0), (-1,-1), 10),
        ('BOTTOMPADDING', (0,0), (-1,-1), 10),
    ]))
    elements.append(risk_table)
    elements.append(Spacer(1, 12))

    # ── CLINICAL PARAMETERS ──
    elements.append(Paragraph("Clinical Parameters", section_style))
    clin_data = [
        ['Parameter', 'Value', 'Parameter', 'Value'],
        ['Age', f"{patient_data['age']} years", 'Diastolic BP', f"{patient_data['diastolic_bp']} mmHg"],
        ['Glucose', f"{patient_data['glucose']} mmol/L", 'Clinical Risk Score', f"{patient_data['clinical_risk_score']}%"],
        ['BMI', f"{patient_data['bmi']} kg/m²", '', ''],
    ]
    clin_table = Table(clin_data, colWidths=[40*mm, 45*mm, 40*mm, 45*mm])
    clin_table.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,0), colors.HexColor('#1F4E79')),
        ('TEXTCOLOR', (0,0), (-1,0), colors.white),
        ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),
        ('FONTSIZE', (0,0), (-1,-1), 8),
        ('GRID', (0,0), (-1,-1), 0.5, colors.HexColor('#DDDDDD')),
        ('BACKGROUND', (0,1), (-1,-1), colors.HexColor('#F8FAFC')),
        ('TOPPADDING', (0,0), (-1,-1), 5),
        ('BOTTOMPADDING', (0,0), (-1,-1), 5),
    ]))
    elements.append(clin_table)
    elements.append(Spacer(1, 12))

    # ── RETINAL IMAGE ANALYSIS ──
    elements.append(Paragraph("Retinal Image Analysis (Grad-CAM)", section_style))
    elements.append(RLImage(gradcam_path, width=70*mm, height=70*mm))
    elements.append(Paragraph(
        f"Grad-CAM heatmap highlights regions the image model identified as most influential in classifying this retina as <b>{patient_data['predicted_stage_label']}</b> (Stage {patient_data['predicted_dr_stage']}), with {patient_data['image_confidence']}% model confidence.",
        body_style))
    elements.append(Spacer(1, 12))

    # ── SHAP EXPLANATION ──
    elements.append(Paragraph("Clinical Risk Explanation (SHAP)", section_style))
    elements.append(RLImage(shap_path, width=140*mm, height=90*mm))
    elements.append(Paragraph(
        "The chart above shows how each clinical parameter contributed to this patient's risk score, in order of impact.",
        body_style))
    elements.append(Spacer(1, 12))

    # ── RECOMMENDATIONS ──
    elements.append(Paragraph("Clinical Recommendations", section_style))
    if risk_level == "LOW":
        recs = [
            "Continue routine annual diabetic retinopathy screening.",
            "Maintain current glycaemic control — glucose levels are within a healthy range.",
            "No urgent ophthalmological referral indicated based on current screening result."
        ]
    elif risk_level == "MODERATE":
        recs = [
            "Schedule ophthalmological review within 3-6 months.",
            "Review glycaemic control and blood pressure management with primary physician.",
            "Repeat screening in 6 months to monitor progression."
        ]
    else:
        recs = [
            "Refer to ophthalmologist for dilated fundus examination within 4 weeks.",
            "Urgent review of glycaemic control indicated.",
            "Repeat screening in 3 months following specialist review."
        ]
    for rec in recs:
        elements.append(Paragraph(f"• {rec}", body_style))
    elements.append(Spacer(1, 14))

    # ── DISCLAIMER ──
    elements.append(Paragraph(
        "This report is generated by an AI screening system and must be reviewed and confirmed by a qualified "
        "clinician before any clinical decision is made. This system is a decision-support tool and is not a "
        "diagnostic device.", disclaimer_style))

    doc.build(elements)
    print(f"Report generated: {output_path}")

# Generate the report
generate_dr_report(
    patient_data=patient_data,
    gradcam_path='/content/report_output/patient_gradcam.png',
    shap_path='/content/report_output/patient_shap.png',
    output_path='/content/report_output/DR_Report_Demo_Patient.pdf'
)

Report generated: /content/report_output/DR_Report_Demo_Patient.pdf


In [53]:
!pip install reportlab -q
print("ReportLab installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 83.2 MB/s eta 0:00:00
ReportLab installed.


In [56]:
def generate_dr_report(patient_data, gradcam_path, shap_path, output_path):
    doc = SimpleDocTemplate(output_path, pagesize=A4,
                             topMargin=15*mm, bottomMargin=15*mm,
                             leftMargin=18*mm, rightMargin=18*mm)

    styles = getSampleStyleSheet()
    title_style = ParagraphStyle('TitleStyle', parent=styles['Heading1'],
                                   fontSize=16, textColor=colors.HexColor('#1F4E79'), spaceAfter=4)
    subtitle_style = ParagraphStyle('SubtitleStyle', parent=styles['Normal'],
                                      fontSize=9, textColor=colors.HexColor('#666666'), spaceAfter=12)
    section_style = ParagraphStyle('SectionStyle', parent=styles['Heading2'],
                                     fontSize=11, textColor=colors.HexColor('#2E75B6'),
                                     spaceBefore=10, spaceAfter=6)
    body_style = ParagraphStyle('BodyStyle', parent=styles['Normal'], fontSize=9, leading=13)
    disclaimer_style = ParagraphStyle('DisclaimerStyle', parent=styles['Normal'],
                                        fontSize=7, textColor=colors.HexColor('#888888'),
                                        fontName='Helvetica-Oblique')

    elements = []
    elements.append(Paragraph("Diabetic Retinopathy Screening Report", title_style))
    elements.append(Paragraph("AI-assisted clinical decision support — for physician review only", subtitle_style))

    report_id = f"DR-{datetime.now().strftime('%Y%m%d')}-{np.random.randint(1000,9999)}"
    elements.append(Paragraph(f"<b>Report ID:</b> {report_id} &nbsp;&nbsp; <b>Generated:</b> {datetime.now().strftime('%d %B %Y, %H:%M')}", body_style))
    elements.append(Spacer(1, 10))

    risk_score = patient_data['clinical_risk_score']
    risk_level = "LOW" if risk_score < 30 else "MODERATE" if risk_score < 60 else "HIGH"
    risk_color = colors.HexColor('#27A745') if risk_level == "LOW" else colors.HexColor('#EF9F27') if risk_level == "MODERATE" else colors.HexColor('#E24B4A')

    risk_table = Table([[f"{risk_score}%", f"{risk_level} RISK\nPredicted: {patient_data['predicted_stage_label']} (Stage {patient_data['predicted_dr_stage']})\nImage model confidence: {patient_data['image_confidence']}%"]], colWidths=[40*mm, 130*mm])
    risk_table.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,-1), colors.HexColor('#FCEBEB') if risk_level=="HIGH" else colors.HexColor('#FFF8E1') if risk_level=="MODERATE" else colors.HexColor('#E8F5E9')),
        ('TEXTCOLOR', (0,0), (0,0), risk_color),
        ('FONTSIZE', (0,0), (0,0), 24), ('FONTNAME', (0,0), (0,0), 'Helvetica-Bold'),
        ('FONTSIZE', (1,0), (1,0), 9), ('VALIGN', (0,0), (-1,-1), 'MIDDLE'),
        ('LEFTPADDING', (0,0), (-1,-1), 12), ('TOPPADDING', (0,0), (-1,-1), 10), ('BOTTOMPADDING', (0,0), (-1,-1), 10),
    ]))
    elements.append(risk_table)
    elements.append(Spacer(1, 12))

    elements.append(Paragraph("Clinical Parameters", section_style))
    clin_data = [
        ['Parameter', 'Value', 'Parameter', 'Value'],
        ['Age', f"{patient_data['age']} years", 'Diastolic BP', f"{patient_data['diastolic_bp']} mmHg"],
        ['Glucose', f"{patient_data['glucose']} mmol/L", 'Clinical Risk Score', f"{patient_data['clinical_risk_score']}%"],
        ['BMI', f"{patient_data['bmi']} kg/m²", '', ''],
    ]
    clin_table = Table(clin_data, colWidths=[40*mm, 45*mm, 40*mm, 45*mm])
    clin_table.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,0), colors.HexColor('#1F4E79')), ('TEXTCOLOR', (0,0), (-1,0), colors.white),
        ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'), ('FONTSIZE', (0,0), (-1,-1), 8),
        ('GRID', (0,0), (-1,-1), 0.5, colors.HexColor('#DDDDDD')), ('BACKGROUND', (0,1), (-1,-1), colors.HexColor('#F8FAFC')),
        ('TOPPADDING', (0,0), (-1,-1), 5), ('BOTTOMPADDING', (0,0), (-1,-1), 5),
    ]))
    elements.append(clin_table)
    elements.append(Spacer(1, 10))

    gradcam_section = [
        Paragraph("Retinal Image Analysis (Grad-CAM)", section_style),
        RLImage(gradcam_path, width=55*mm, height=55*mm),
        Paragraph(f"Grad-CAM heatmap highlights regions the image model identified as most influential in classifying this retina as <b>{patient_data['predicted_stage_label']}</b> (Stage {patient_data['predicted_dr_stage']}), with {patient_data['image_confidence']}% model confidence.", body_style)
    ]
    elements.append(KeepTogether(gradcam_section))
    elements.append(Spacer(1, 10))

    shap_section = [
        Paragraph("Clinical Risk Explanation (SHAP)", section_style),
        RLImage(shap_path, width=130*mm, height=84*mm),
        Paragraph("The chart above shows how each clinical parameter contributed to this patient's risk score, in order of impact.", body_style)
    ]
    elements.append(KeepTogether(shap_section))
    elements.append(Spacer(1, 10))

    elements.append(Paragraph("Clinical Recommendations", section_style))
    if risk_level == "LOW":
        recs = ["Continue routine annual diabetic retinopathy screening.",
                "Maintain current glycaemic control — glucose levels are within a healthy range.",
                "No urgent ophthalmological referral indicated based on current screening result."]
    elif risk_level == "MODERATE":
        recs = ["Schedule ophthalmological review within 3-6 months.",
                "Review glycaemic control and blood pressure management with primary physician.",
                "Repeat screening in 6 months to monitor progression."]
    else:
        recs = ["Refer to ophthalmologist for dilated fundus examination within 4 weeks.",
                "Urgent review of glycaemic control indicated.",
                "Repeat screening in 3 months following specialist review."]
    for rec in recs:
        elements.append(Paragraph(f"• {rec}", body_style))
    elements.append(Spacer(1, 14))

    elements.append(Paragraph("This report is generated by an AI screening system and must be reviewed and confirmed by a qualified clinician before any clinical decision is made. This system is a decision-support tool and is not a diagnostic device.", disclaimer_style))

    doc.build(elements)
    print(f"Report generated: {output_path}")

generate_dr_report(
    patient_data=patient_data,
    gradcam_path='/content/report_output/patient_gradcam.png',
    shap_path='/content/report_output/patient_shap.png',
    output_path='/content/report_output/DR_Report_Demo_Patient_v2.pdf'
)

Report generated: /content/report_output/DR_Report_Demo_Patient_v2.pdf


In [57]:
!pip install reportlab -q

from reportlab.lib.pagesizes import A4
from reportlab.lib.units import mm
from reportlab.lib import colors
from reportlab.platypus import (BaseDocTemplate, PageTemplate, Frame, Paragraph, Spacer,
                                  Table, TableStyle, Image as RLImage, PageBreak, KeepTogether)
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER, TA_LEFT, TA_RIGHT
from reportlab.graphics.shapes import Drawing, Rect, String, Circle, Line
from reportlab.graphics import renderPDF
from datetime import datetime
import math

# ── PALETTE ──
NAVY = colors.HexColor('#0A2342')
TEAL = colors.HexColor('#0D9488')
TEAL_LIGHT = colors.HexColor('#CCFBF1')
INK = colors.HexColor('#1E293B')
GRAY = colors.HexColor('#64748B')
GRAY_LIGHT = colors.HexColor('#E2E8F0')
BG_LIGHT = colors.HexColor('#F8FAFC')
GREEN = colors.HexColor('#059669')
GREEN_BG = colors.HexColor('#ECFDF5')
AMBER = colors.HexColor('#D97706')
AMBER_BG = colors.HexColor('#FFFBEB')
RED = colors.HexColor('#DC2626')
RED_BG = colors.HexColor('#FEF2F2')

REPORT_ID = f"DR-{datetime.now().strftime('%Y%m%d')}-{__import__('random').randint(1000,9999)}"
GEN_TIME = datetime.now().strftime('%d %B %Y, %H:%M')

# ── HEADER / FOOTER drawn on every page ──
def draw_header_footer(canvas, doc):
    canvas.saveState()
    page_w, page_h = A4

    # Top navy band
    canvas.setFillColor(NAVY)
    canvas.rect(0, page_h - 22*mm, page_w, 22*mm, fill=1, stroke=0)

    # Teal accent strip
    canvas.setFillColor(TEAL)
    canvas.rect(0, page_h - 22*mm, 3*mm, 22*mm, fill=1, stroke=0)

    canvas.setFillColor(colors.white)
    canvas.setFont('Helvetica-Bold', 13)
    canvas.drawString(12*mm, page_h - 10*mm, "DIABETIC RETINOPATHY SCREENING REPORT")
    canvas.setFont('Helvetica', 8)
    canvas.setFillColor(TEAL_LIGHT)
    canvas.drawString(12*mm, page_h - 16*mm, "AI-Assisted Clinical Decision Support  \u2014  For Physician Review Only")

    canvas.setFont('Helvetica', 7.5)
    canvas.setFillColor(colors.white)
    canvas.drawRightString(page_w - 12*mm, page_h - 9*mm, f"Report ID: {REPORT_ID}")
    canvas.drawRightString(page_w - 12*mm, page_h - 15*mm, f"Generated: {GEN_TIME}")

    # Footer
    canvas.setFillColor(GRAY)
    canvas.setFont('Helvetica', 7)
    canvas.drawString(12*mm, 10*mm, "DR Detection System \u2014 Clinical Decision Support Tool")
    canvas.drawRightString(page_w - 12*mm, 10*mm, f"Page {doc.page} of 2")
    canvas.setStrokeColor(GRAY_LIGHT)
    canvas.setLineWidth(0.5)
    canvas.line(12*mm, 13*mm, page_w - 12*mm, 13*mm)

    canvas.restoreState()

# ── DOC + FRAME SETUP ──
def build_doc(output_path):
    doc = BaseDocTemplate(output_path, pagesize=A4,
                           topMargin=28*mm, bottomMargin=18*mm,
                           leftMargin=12*mm, rightMargin=12*mm)
    frame = Frame(doc.leftMargin, doc.bottomMargin, doc.width, doc.height, id='normal')
    template = PageTemplate(id='report', frames=[frame], onPage=draw_header_footer)
    doc.addPageTemplates([template])
    return doc

# ── STYLES ──
styles = getSampleStyleSheet()
section_style = ParagraphStyle('Section', fontSize=11.5, textColor=NAVY, fontName='Helvetica-Bold',
                                 spaceBefore=4, spaceAfter=8)
label_style = ParagraphStyle('Label', fontSize=7.5, textColor=GRAY, fontName='Helvetica-Bold', leading=10)
value_style = ParagraphStyle('Value', fontSize=10.5, textColor=INK, fontName='Helvetica-Bold', leading=13)
body_style = ParagraphStyle('Body', fontSize=9, textColor=INK, leading=14)
caption_style = ParagraphStyle('Caption', fontSize=8, textColor=GRAY, leading=12, fontName='Helvetica-Oblique')
disclaimer_style = ParagraphStyle('Disclaimer', fontSize=7, textColor=GRAY, leading=10, fontName='Helvetica-Oblique')

# ── RISK GAUGE (drawn vector graphic, not just text) ──
def draw_risk_gauge(risk_pct, risk_level, risk_color_hex):
    d = Drawing(60*mm, 32*mm)
    cx, cy, r = 30*mm, 4*mm, 22*mm

    # Background arc segments (green/amber/red zones) drawn as thick arc via wedge-like lines
    import reportlab.graphics.shapes as shapes
    from reportlab.graphics.shapes import Wedge

    zones = [(0, 30, colors.HexColor('#D1FAE5')), (30, 60, colors.HexColor('#FEF3C7')), (60, 100, colors.HexColor('#FEE2E2'))]
    for start, end, col in zones:
        a1 = 180 - (start/100*180)
        a2 = 180 - (end/100*180)
        w = Wedge(cx, cy, r, a2, a1, radius1=r-4*mm, fillColor=col, strokeColor=None)
        d.add(w)

    # Needle
    angle_deg = 180 - (risk_pct/100*180)
    angle_rad = math.radians(angle_deg)
    needle_len = r - 2*mm
    nx = cx + needle_len * math.cos(angle_rad)
    ny = cy + needle_len * math.sin(angle_rad)
    d.add(Line(cx, cy, nx, ny, strokeColor=NAVY, strokeWidth=2))
    d.add(Circle(cx, cy, 2*mm, fillColor=NAVY, strokeColor=None))

    # Value text
    d.add(String(cx, cy + 6*mm, f"{risk_pct}%", fontSize=18, fontName='Helvetica-Bold',
                  fillColor=colors.HexColor(risk_color_hex), textAnchor='middle'))
    d.add(String(cx, cy - 3*mm, risk_level, fontSize=8, fontName='Helvetica-Bold',
                  fillColor=GRAY, textAnchor='middle'))

    return d

# ── MAIN BUILD FUNCTION ──
def generate_dr_report_v2(patient_data, gradcam_path, shap_path, output_path):
    doc = build_doc(output_path)
    elements = []

    risk_score = patient_data['clinical_risk_score']
    risk_level = "LOW RISK" if risk_score < 30 else "MODERATE RISK" if risk_score < 60 else "HIGH RISK"
    risk_hex = '#059669' if risk_score < 30 else '#D97706' if risk_score < 60 else '#DC2626'
    risk_bg = GREEN_BG if risk_score < 30 else AMBER_BG if risk_score < 60 else RED_BG
    risk_border = GREEN if risk_score < 30 else AMBER if risk_score < 60 else RED

    # ── PATIENT ID STRIP ──
    id_table = Table([[
        Paragraph(f"<font color='#64748B' size=7><b>PATIENT ID</b></font><br/><font size=10>PT-{REPORT_ID[-4:]}</font>", body_style),
        Paragraph(f"<font color='#64748B' size=7><b>SCREENING DATE</b></font><br/><font size=10>{datetime.now().strftime('%d %b %Y')}</font>", body_style),
        Paragraph(f"<font color='#64748B' size=7><b>SYSTEM</b></font><br/><font size=10>Multi-modal AI v1.0</font>", body_style),
    ]], colWidths=[58*mm, 58*mm, 58*mm])
    id_table.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,-1), BG_LIGHT),
        ('BOX', (0,0), (-1,-1), 0.5, GRAY_LIGHT),
        ('LINEAFTER', (0,0), (0,0), 0.5, GRAY_LIGHT),
        ('LINEAFTER', (1,0), (1,0), 0.5, GRAY_LIGHT),
        ('TOPPADDING', (0,0), (-1,-1), 8), ('BOTTOMPADDING', (0,0), (-1,-1), 8),
        ('LEFTPADDING', (0,0), (-1,-1), 10),
    ]))
    elements.append(id_table)
    elements.append(Spacer(1, 10))

    # ── RISK CARD WITH GAUGE ──
    gauge = draw_risk_gauge(risk_score, risk_level, risk_hex)
    stage_text = f"""<font size=13 color='#0A2342'><b>{patient_data['predicted_stage_label']}</b></font>
    <font size=9 color='#64748B'>(Stage {patient_data['predicted_dr_stage']})</font><br/><br/>
    <font size=8 color='#64748B'>IMAGE MODEL CONFIDENCE</font><br/>
    <font size=11 color='#0A2342'><b>{patient_data['image_confidence']}%</b></font>"""

    risk_card = Table([[gauge, Paragraph(stage_text, body_style)]], colWidths=[65*mm, 109*mm])
    risk_card.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,-1), risk_bg),
        ('BOX', (0,0), (-1,-1), 1, risk_border),
        ('VALIGN', (0,0), (-1,-1), 'MIDDLE'),
        ('TOPPADDING', (0,0), (-1,-1), 14), ('BOTTOMPADDING', (0,0), (-1,-1), 14),
        ('LEFTPADDING', (1,0), (1,0), 14),
    ]))
    elements.append(risk_card)
    elements.append(Spacer(1, 14))

    # ── CLINICAL PARAMETERS ──
    elements.append(Paragraph("CLINICAL PARAMETERS", section_style))
    param_rows = [
        ('Age', f"{patient_data['age']} yrs", 'Diastolic BP', f"{patient_data['diastolic_bp']} mmHg"),
        ('Glucose', f"{patient_data['glucose']} mmol/L", 'BMI', f"{patient_data['bmi']} kg/m\u00b2"),
    ]
    clin_data = [[Paragraph(f"<font size=7 color='#64748B'><b>{a}</b></font>", body_style), Paragraph(f"<font size=10>{b}</font>", body_style),
                  Paragraph(f"<font size=7 color='#64748B'><b>{c}</b></font>", body_style), Paragraph(f"<font size=10>{d}</font>", body_style)]
                 for a,b,c,d in param_rows]
    clin_table = Table(clin_data, colWidths=[30*mm, 44*mm, 30*mm, 44*mm])
    clin_table.setStyle(TableStyle([
        ('GRID', (0,0), (-1,-1), 0.5, GRAY_LIGHT),
        ('BACKGROUND', (0,0), (0,-1), BG_LIGHT), ('BACKGROUND', (2,0), (2,-1), BG_LIGHT),
        ('TOPPADDING', (0,0), (-1,-1), 7), ('BOTTOMPADDING', (0,0), (-1,-1), 7),
        ('LEFTPADDING', (0,0), (-1,-1), 8),
    ]))
    elements.append(clin_table)
    elements.append(Spacer(1, 16))

    # ── GRAD-CAM ──
    elements.append(Paragraph("RETINAL IMAGE ANALYSIS \u2014 GRAD-CAM", section_style))
    img_card = Table([[RLImage(gradcam_path, width=60*mm, height=60*mm)]], colWidths=[174*mm])
    img_card.setStyle(TableStyle([('ALIGN', (0,0), (-1,-1), 'CENTER'), ('BOX', (0,0), (-1,-1), 0.5, GRAY_LIGHT), ('TOPPADDING',(0,0),(-1,-1),8), ('BOTTOMPADDING',(0,0),(-1,-1),8)]))
    elements.append(img_card)
    elements.append(Spacer(1, 6))
    elements.append(Paragraph(f"Heatmap highlights regions the model identified as most influential in classifying this retina as "
                                f"<b>{patient_data['predicted_stage_label']}</b> (Stage {patient_data['predicted_dr_stage']}), "
                                f"{patient_data['image_confidence']}% confidence.", caption_style))

    elements.append(PageBreak())

    # ── SHAP ──
    elements.append(Paragraph("CLINICAL RISK EXPLANATION \u2014 SHAP", section_style))
    shap_card = Table([[RLImage(shap_path, width=150*mm, height=95*mm)]], colWidths=[174*mm])
    shap_card.setStyle(TableStyle([('ALIGN', (0,0), (-1,-1), 'CENTER'), ('BOX', (0,0), (-1,-1), 0.5, GRAY_LIGHT), ('TOPPADDING',(0,0),(-1,-1),8), ('BOTTOMPADDING',(0,0),(-1,-1),8)]))
    elements.append(shap_card)
    elements.append(Spacer(1, 6))
    elements.append(Paragraph("Each bar shows how a clinical parameter shifted this patient's risk score, ranked by impact.", caption_style))
    elements.append(Spacer(1, 16))

    # ── RECOMMENDATIONS ──
    elements.append(Paragraph("CLINICAL RECOMMENDATIONS", section_style))
    if risk_score < 30:
        recs = ["Continue routine annual diabetic retinopathy screening.",
                "Maintain current glycaemic control \u2014 glucose levels are within a healthy range.",
                "No urgent ophthalmological referral indicated based on current screening result."]
        rec_color = GREEN
    elif risk_score < 60:
        recs = ["Schedule ophthalmological review within 3\u20136 months.",
                "Review glycaemic control and blood pressure management with primary physician.",
                "Repeat screening in 6 months to monitor progression."]
        rec_color = AMBER
    else:
        recs = ["Refer to ophthalmologist for dilated fundus examination within 4 weeks.",
                "Urgent review of glycaemic control indicated.",
                "Repeat screening in 3 months following specialist review."]
        rec_color = RED

    rec_rows = []
    for rec in recs:
        rec_rows.append([Paragraph(f"<font color='{rec_color.hexval()}'>\u25CF</font>", body_style), Paragraph(rec, body_style)])
    rec_table = Table(rec_rows, colWidths=[6*mm, 168*mm])
    rec_table.setStyle(TableStyle([('VALIGN',(0,0),(-1,-1),'TOP'), ('TOPPADDING',(0,0),(-1,-1),3), ('BOTTOMPADDING',(0,0),(-1,-1),3)]))
    elements.append(rec_table)
    elements.append(Spacer(1, 18))

    # ── DISCLAIMER ──
    disclaimer_box = Table([[Paragraph(
        "<b>Disclaimer:</b> This report is generated by an AI screening system and must be reviewed and confirmed "
        "by a qualified clinician before any clinical decision is made. This system is a decision-support tool "
        "and is not a diagnostic device.", disclaimer_style)]], colWidths=[174*mm])
    disclaimer_box.setStyle(TableStyle([('BACKGROUND',(0,0),(-1,-1), BG_LIGHT), ('BOX',(0,0),(-1,-1),0.5,GRAY_LIGHT),
                                          ('TOPPADDING',(0,0),(-1,-1),8), ('BOTTOMPADDING',(0,0),(-1,-1),8), ('LEFTPADDING',(0,0),(-1,-1),8)]))
    elements.append(disclaimer_box)

    doc.build(elements)
    print(f"Professional report generated: {output_path}")

# Clear old outputs and generate fresh
import os
for f in os.listdir('/content/report_output'):
    if f.endswith('.pdf'):
        os.remove(f'/content/report_output/{f}')

generate_dr_report_v2(
    patient_data=patient_data,
    gradcam_path='/content/report_output/patient_gradcam.png',
    shap_path='/content/report_output/patient_shap.png',
    output_path='/content/report_output/DR_Report_Professional.pdf'
)

Professional report generated: /content/report_output/DR_Report_Professional.pdf


In [58]:
def generate_dr_report_v3(patient_data, gradcam_path=None, shap_path=None, output_path=None):
    """
    Generates a DR screening report that adapts to available data:
    - If only clinical data provided (no gradcam_path): Mode 1 report (SHAP only)
    - If only image data provided (no shap_path): Mode 2 report (Grad-CAM only)
    - If both provided: Mode 3 report (full fusion report)
    """
    has_clinical = shap_path is not None
    has_image = gradcam_path is not None

    if not has_clinical and not has_image:
        raise ValueError("At least one of clinical data or image data must be provided.")

    mode = "Mode 3 — Fusion" if (has_clinical and has_image) else "Mode 1 — Clinical Only" if has_clinical else "Mode 2 — Image Only"

    doc = build_doc(output_path)
    elements = []

    # ── Determine risk score/level based on what's available ──
    if has_clinical:
        risk_score = patient_data['clinical_risk_score']
    else:
        # Image-only mode: derive a severity-based "risk" purely from predicted stage
        risk_score = patient_data['predicted_dr_stage'] * 25  # rough 0-100 scale from stage 0-4

    risk_level = "LOW RISK" if risk_score < 30 else "MODERATE RISK" if risk_score < 60 else "HIGH RISK"
    risk_hex = '#059669' if risk_score < 30 else '#D97706' if risk_score < 60 else '#DC2626'
    risk_bg = GREEN_BG if risk_score < 30 else AMBER_BG if risk_score < 60 else RED_BG
    risk_border = GREEN if risk_score < 30 else AMBER if risk_score < 60 else RED

    # ── Patient ID strip — now shows the mode used ──
    id_table = Table([[
        Paragraph(f"<font color='#64748B' size=7><b>PATIENT ID</b></font><br/><font size=10>PT-{REPORT_ID[-4:]}</font>", body_style),
        Paragraph(f"<font color='#64748B' size=7><b>SCREENING DATE</b></font><br/><font size=10>{datetime.now().strftime('%d %b %Y')}</font>", body_style),
        Paragraph(f"<font color='#64748B' size=7><b>MODE USED</b></font><br/><font size=10>{mode}</font>", body_style),
    ]], colWidths=[58*mm, 58*mm, 58*mm])
    id_table.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,-1), BG_LIGHT), ('BOX', (0,0), (-1,-1), 0.5, GRAY_LIGHT),
        ('LINEAFTER', (0,0), (0,0), 0.5, GRAY_LIGHT), ('LINEAFTER', (1,0), (1,0), 0.5, GRAY_LIGHT),
        ('TOPPADDING', (0,0), (-1,-1), 8), ('BOTTOMPADDING', (0,0), (-1,-1), 8), ('LEFTPADDING', (0,0), (-1,-1), 10),
    ]))
    elements.append(id_table)
    elements.append(Spacer(1, 10))

    # ── Risk card — text differs depending on mode ──
    gauge = draw_risk_gauge(risk_score, risk_level, risk_hex)
    if has_image:
        stage_text = f"""<font size=13 color='#0A2342'><b>{patient_data['predicted_stage_label']}</b></font>
        <font size=9 color='#64748B'>(Stage {patient_data['predicted_dr_stage']})</font><br/><br/>
        <font size=8 color='#64748B'>IMAGE MODEL CONFIDENCE</font><br/>
        <font size=11 color='#0A2342'><b>{patient_data['image_confidence']}%</b></font>"""
    else:
        stage_text = f"""<font size=13 color='#0A2342'><b>Clinical Risk Assessment</b></font><br/><br/>
        <font size=8 color='#64748B'>BASED ON</font><br/>
        <font size=10 color='#0A2342'>13 clinical parameters (no retinal image provided)</font>"""

    risk_card = Table([[gauge, Paragraph(stage_text, body_style)]], colWidths=[65*mm, 109*mm])
    risk_card.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,-1), risk_bg), ('BOX', (0,0), (-1,-1), 1, risk_border),
        ('VALIGN', (0,0), (-1,-1), 'MIDDLE'), ('TOPPADDING', (0,0), (-1,-1), 14), ('BOTTOMPADDING', (0,0), (-1,-1), 14),
        ('LEFTPADDING', (1,0), (1,0), 14),
    ]))
    elements.append(risk_card)
    elements.append(Spacer(1, 14))

    # ── Clinical parameters — only shown if clinical data exists ──
    if has_clinical:
        elements.append(Paragraph("CLINICAL PARAMETERS", section_style))
        param_rows = [
            ('Age', f"{patient_data['age']} yrs", 'Diastolic BP', f"{patient_data['diastolic_bp']} mmHg"),
            ('Glucose', f"{patient_data['glucose']} mmol/L", 'BMI', f"{patient_data['bmi']} kg/m\u00b2"),
        ]
        clin_data = [[Paragraph(f"<font size=7 color='#64748B'><b>{a}</b></font>", body_style), Paragraph(f"<font size=10>{b}</font>", body_style),
                      Paragraph(f"<font size=7 color='#64748B'><b>{c}</b></font>", body_style), Paragraph(f"<font size=10>{d}</font>", body_style)]
                     for a,b,c,d in param_rows]
        clin_table = Table(clin_data, colWidths=[30*mm, 44*mm, 30*mm, 44*mm])
        clin_table.setStyle(TableStyle([
            ('GRID', (0,0), (-1,-1), 0.5, GRAY_LIGHT), ('BACKGROUND', (0,0), (0,-1), BG_LIGHT), ('BACKGROUND', (2,0), (2,-1), BG_LIGHT),
            ('TOPPADDING', (0,0), (-1,-1), 7), ('BOTTOMPADDING', (0,0), (-1,-1), 7), ('LEFTPADDING', (0,0), (-1,-1), 8),
        ]))
        elements.append(clin_table)
        elements.append(Spacer(1, 16))
    else:
        elements.append(Paragraph("CLINICAL PARAMETERS", section_style))
        elements.append(Paragraph("<i>No clinical data provided for this screening. Risk assessment based on retinal image analysis only.</i>", caption_style))
        elements.append(Spacer(1, 16))

    # ── Grad-CAM section — only if image was provided ──
    if has_image:
        elements.append(Paragraph("RETINAL IMAGE ANALYSIS \u2014 GRAD-CAM", section_style))
        img_card = Table([[RLImage(gradcam_path, width=60*mm, height=60*mm)]], colWidths=[174*mm])
        img_card.setStyle(TableStyle([('ALIGN', (0,0), (-1,-1), 'CENTER'), ('BOX', (0,0), (-1,-1), 0.5, GRAY_LIGHT), ('TOPPADDING',(0,0),(-1,-1),8), ('BOTTOMPADDING',(0,0),(-1,-1),8)]))
        elements.append(img_card)
        elements.append(Spacer(1, 6))
        elements.append(Paragraph(f"Heatmap highlights regions the model identified as most influential in classifying this retina as "
                                    f"<b>{patient_data['predicted_stage_label']}</b> (Stage {patient_data['predicted_dr_stage']}), "
                                    f"{patient_data['image_confidence']}% confidence.", caption_style))

    if has_clinical and has_image:
        elements.append(PageBreak())
    elif has_image and not has_clinical:
        elements.append(Spacer(1, 16))

    # ── SHAP section — only if clinical data was provided ──
    if has_clinical:
        elements.append(Paragraph("CLINICAL RISK EXPLANATION \u2014 SHAP", section_style))
        shap_card = Table([[RLImage(shap_path, width=150*mm, height=95*mm)]], colWidths=[174*mm])
        shap_card.setStyle(TableStyle([('ALIGN', (0,0), (-1,-1), 'CENTER'), ('BOX', (0,0), (-1,-1), 0.5, GRAY_LIGHT), ('TOPPADDING',(0,0),(-1,-1),8), ('BOTTOMPADDING',(0,0),(-1,-1),8)]))
        elements.append(shap_card)
        elements.append(Spacer(1, 6))
        elements.append(Paragraph("Each bar shows how a clinical parameter shifted this patient's risk score, ranked by impact.", caption_style))
        elements.append(Spacer(1, 16))

    # ── Recommendations — same logic, always shown ──
    elements.append(Paragraph("CLINICAL RECOMMENDATIONS", section_style))
    if risk_score < 30:
        recs = ["Continue routine annual diabetic retinopathy screening.",
                "Maintain current glycaemic control \u2014 values are within a healthy range." if has_clinical else "Continue standard diabetic eye care schedule.",
                "No urgent ophthalmological referral indicated based on current screening result."]
        rec_color = GREEN
    elif risk_score < 60:
        recs = ["Schedule ophthalmological review within 3\u20136 months.",
                "Review glycaemic control and blood pressure management with primary physician." if has_clinical else "Recommend clinical data collection for a more complete risk assessment.",
                "Repeat screening in 6 months to monitor progression."]
        rec_color = AMBER
    else:
        recs = ["Refer to ophthalmologist for dilated fundus examination within 4 weeks.",
                "Urgent review of glycaemic control indicated." if has_clinical else "Recommend full clinical work-up to complement this image-based finding.",
                "Repeat screening in 3 months following specialist review."]
        rec_color = RED

    if not has_image:
        recs.insert(0, "No retinal image was provided \u2014 this assessment is based on clinical risk factors only. A retinal examination is recommended to confirm DR status.")

    rec_rows = [[Paragraph(f"<font color='{rec_color.hexval()}'>\u25CF</font>", body_style), Paragraph(rec, body_style)] for rec in recs]
    rec_table = Table(rec_rows, colWidths=[6*mm, 168*mm])
    rec_table.setStyle(TableStyle([('VALIGN',(0,0),(-1,-1),'TOP'), ('TOPPADDING',(0,0),(-1,-1),3), ('BOTTOMPADDING',(0,0),(-1,-1),3)]))
    elements.append(rec_table)
    elements.append(Spacer(1, 18))

    disclaimer_box = Table([[Paragraph(
        "<b>Disclaimer:</b> This report is generated by an AI screening system and must be reviewed and confirmed "
        "by a qualified clinician before any clinical decision is made. This system is a decision-support tool "
        "and is not a diagnostic device.", disclaimer_style)]], colWidths=[174*mm])
    disclaimer_box.setStyle(TableStyle([('BACKGROUND',(0,0),(-1,-1), BG_LIGHT), ('BOX',(0,0),(-1,-1),0.5,GRAY_LIGHT),
                                          ('TOPPADDING',(0,0),(-1,-1),8), ('BOTTOMPADDING',(0,0),(-1,-1),8), ('LEFTPADDING',(0,0),(-1,-1),8)]))
    elements.append(disclaimer_box)

    doc.build(elements)
    print(f"Report generated ({mode}): {output_path}")

In [59]:
# Mode 1 — clinical only (no image)
generate_dr_report_v3(
    patient_data=patient_data,
    shap_path='/content/report_output/patient_shap.png',
    gradcam_path=None,
    output_path='/content/report_output/DR_Report_Mode1_ClinicalOnly.pdf'
)

# Mode 2 — image only (no clinical data)
generate_dr_report_v3(
    patient_data=patient_data,
    gradcam_path='/content/report_output/patient_gradcam.png',
    shap_path=None,
    output_path='/content/report_output/DR_Report_Mode2_ImageOnly.pdf'
)

# Mode 3 — both (full fusion report, what you already saw)
generate_dr_report_v3(
    patient_data=patient_data,
    gradcam_path='/content/report_output/patient_gradcam.png',
    shap_path='/content/report_output/patient_shap.png',
    output_path='/content/report_output/DR_Report_Mode3_Fusion.pdf'
)


Report generated (Mode 1 — Clinical Only): /content/report_output/DR_Report_Mode1_ClinicalOnly.pdf
Report generated (Mode 2 — Image Only): /content/report_output/DR_Report_Mode2_ImageOnly.pdf
Report generated (Mode 3 — Fusion): /content/report_output/DR_Report_Mode3_Fusion.pdf


In [60]:
risk_score = patient_data['predicted_dr_stage'] * 25  # rough 0-100 scale from stage 0-4

In [61]:
!pip install reportlab -q

from reportlab.lib.pagesizes import A4
from reportlab.lib.units import mm
from reportlab.lib import colors
from reportlab.platypus import (BaseDocTemplate, PageTemplate, Frame, Paragraph, Spacer,
                                  Table, TableStyle, Image as RLImage, PageBreak)
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.graphics.shapes import Drawing, String, Circle, Line, Wedge
from datetime import datetime
import math
import random

# ── PALETTE ──
NAVY = colors.HexColor('#0A2342')
TEAL = colors.HexColor('#0D9488')
TEAL_LIGHT = colors.HexColor('#CCFBF1')
INK = colors.HexColor('#1E293B')
GRAY = colors.HexColor('#64748B')
GRAY_LIGHT = colors.HexColor('#E2E8F0')
BG_LIGHT = colors.HexColor('#F8FAFC')
GREEN = colors.HexColor('#059669')
GREEN_BG = colors.HexColor('#ECFDF5')
AMBER = colors.HexColor('#D97706')
AMBER_BG = colors.HexColor('#FFFBEB')
RED = colors.HexColor('#DC2626')
RED_BG = colors.HexColor('#FEF2F2')

# ── STYLES ──
styles = getSampleStyleSheet()
section_style = ParagraphStyle('Section', fontSize=11.5, textColor=NAVY, fontName='Helvetica-Bold', spaceBefore=4, spaceAfter=8)
label_style = ParagraphStyle('Label', fontSize=7.5, textColor=GRAY, fontName='Helvetica-Bold', leading=10)
value_style = ParagraphStyle('Value', fontSize=10.5, textColor=INK, fontName='Helvetica-Bold', leading=13)
body_style = ParagraphStyle('Body', fontSize=9, textColor=INK, leading=14)
caption_style = ParagraphStyle('Caption', fontSize=8, textColor=GRAY, leading=12, fontName='Helvetica-Oblique')
disclaimer_style = ParagraphStyle('Disclaimer', fontSize=7, textColor=GRAY, leading=10, fontName='Helvetica-Oblique')


def draw_risk_gauge(risk_pct, risk_level, risk_color_hex):
    d = Drawing(60*mm, 32*mm)
    cx, cy, r = 30*mm, 4*mm, 22*mm

    zones = [(0, 30, colors.HexColor('#D1FAE5')), (30, 60, colors.HexColor('#FEF3C7')), (60, 100, colors.HexColor('#FEE2E2'))]
    for start, end, col in zones:
        a1 = 180 - (start/100*180)
        a2 = 180 - (end/100*180)
        w = Wedge(cx, cy, r, a2, a1, radius1=r-4*mm, fillColor=col, strokeColor=None)
        d.add(w)

    angle_deg = 180 - (risk_pct/100*180)
    angle_rad = math.radians(angle_deg)
    needle_len = r - 2*mm
    nx = cx + needle_len * math.cos(angle_rad)
    ny = cy + needle_len * math.sin(angle_rad)
    d.add(Line(cx, cy, nx, ny, strokeColor=NAVY, strokeWidth=2))
    d.add(Circle(cx, cy, 2*mm, fillColor=NAVY, strokeColor=None))

    d.add(String(cx, cy + 6*mm, f"{risk_pct}%", fontSize=18, fontName='Helvetica-Bold',
                  fillColor=colors.HexColor(risk_color_hex), textAnchor='middle'))
    d.add(String(cx, cy - 3*mm, risk_level, fontSize=8, fontName='Helvetica-Bold',
                  fillColor=GRAY, textAnchor='middle'))
    return d


def draw_header_footer_factory(report_id, gen_time):
    def draw_header_footer(canvas, doc):
        canvas.saveState()
        page_w, page_h = A4

        canvas.setFillColor(NAVY)
        canvas.rect(0, page_h - 22*mm, page_w, 22*mm, fill=1, stroke=0)
        canvas.setFillColor(TEAL)
        canvas.rect(0, page_h - 22*mm, 3*mm, 22*mm, fill=1, stroke=0)

        canvas.setFillColor(colors.white)
        canvas.setFont('Helvetica-Bold', 13)
        canvas.drawString(12*mm, page_h - 10*mm, "DIABETIC RETINOPATHY SCREENING REPORT")
        canvas.setFont('Helvetica', 8)
        canvas.setFillColor(TEAL_LIGHT)
        canvas.drawString(12*mm, page_h - 16*mm, "AI-Assisted Clinical Decision Support  \u2014  For Physician Review Only")

        canvas.setFont('Helvetica', 7.5)
        canvas.setFillColor(colors.white)
        canvas.drawRightString(page_w - 12*mm, page_h - 9*mm, f"Report ID: {report_id}")
        canvas.drawRightString(page_w - 12*mm, page_h - 15*mm, f"Generated: {gen_time}")

        canvas.setFillColor(GRAY)
        canvas.setFont('Helvetica', 7)
        canvas.drawString(12*mm, 10*mm, "DR Detection System \u2014 Clinical Decision Support Tool")
        canvas.drawRightString(page_w - 12*mm, 10*mm, f"Page {doc.page}")
        canvas.setStrokeColor(GRAY_LIGHT)
        canvas.setLineWidth(0.5)
        canvas.line(12*mm, 13*mm, page_w - 12*mm, 13*mm)

        canvas.restoreState()
    return draw_header_footer


def build_doc(output_path, report_id, gen_time):
    doc = BaseDocTemplate(output_path, pagesize=A4,
                           topMargin=28*mm, bottomMargin=18*mm,
                           leftMargin=12*mm, rightMargin=12*mm)
    frame = Frame(doc.leftMargin, doc.bottomMargin, doc.width, doc.height, id='normal')
    template = PageTemplate(id='report', frames=[frame], onPage=draw_header_footer_factory(report_id, gen_time))
    doc.addPageTemplates([template])
    return doc


def generate_dr_report_v3(patient_data, gradcam_path=None, shap_path=None, output_path=None):
    """
    Generates a DR screening report that adapts to available data:
    - Mode 1 (clinical only): shap_path given, gradcam_path=None
    - Mode 2 (image only): gradcam_path given, shap_path=None
    - Mode 3 (fusion): both given
    """
    has_clinical = shap_path is not None
    has_image = gradcam_path is not None

    if not has_clinical and not has_image:
        raise ValueError("At least one of clinical data or image data must be provided.")

    mode = "Mode 3 \u2014 Fusion" if (has_clinical and has_image) else "Mode 1 \u2014 Clinical Only" if has_clinical else "Mode 2 \u2014 Image Only"

    report_id = f"DR-{datetime.now().strftime('%Y%m%d')}-{random.randint(1000,9999)}"
    gen_time = datetime.now().strftime('%d %B %Y, %H:%M')

    doc = build_doc(output_path, report_id, gen_time)
    elements = []

    # ── Risk score logic — differs by mode ──
    if has_clinical:
        risk_score = patient_data['clinical_risk_score']
    else:
        # Image-only: severity stage weighted by model confidence (more honest than stage alone)
        risk_score = round((patient_data['predicted_dr_stage'] / 4.0) * 100 * (patient_data['image_confidence'] / 100), 1)

    risk_level = "LOW RISK" if risk_score < 30 else "MODERATE RISK" if risk_score < 60 else "HIGH RISK"
    risk_hex = '#059669' if risk_score < 30 else '#D97706' if risk_score < 60 else '#DC2626'
    risk_bg = GREEN_BG if risk_score < 30 else AMBER_BG if risk_score < 60 else RED_BG
    risk_border = GREEN if risk_score < 30 else AMBER if risk_score < 60 else RED

    # ── Patient ID strip ──
    id_table = Table([[
        Paragraph(f"<font color='#64748B' size=7><b>PATIENT ID</b></font><br/><font size=10>PT-{report_id[-4:]}</font>", body_style),
        Paragraph(f"<font color='#64748B' size=7><b>SCREENING DATE</b></font><br/><font size=10>{datetime.now().strftime('%d %b %Y')}</font>", body_style),
        Paragraph(f"<font color='#64748B' size=7><b>MODE USED</b></font><br/><font size=10>{mode}</font>", body_style),
    ]], colWidths=[58*mm, 58*mm, 58*mm])
    id_table.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,-1), BG_LIGHT), ('BOX', (0,0), (-1,-1), 0.5, GRAY_LIGHT),
        ('LINEAFTER', (0,0), (0,0), 0.5, GRAY_LIGHT), ('LINEAFTER', (1,0), (1,0), 0.5, GRAY_LIGHT),
        ('TOPPADDING', (0,0), (-1,-1), 8), ('BOTTOMPADDING', (0,0), (-1,-1), 8), ('LEFTPADDING', (0,0), (-1,-1), 10),
    ]))
    elements.append(id_table)
    elements.append(Spacer(1, 10))

    # ── Risk card ──
    gauge = draw_risk_gauge(risk_score, risk_level, risk_hex)
    if has_image:
        stage_text = f"""<font size=13 color='#0A2342'><b>{patient_data['predicted_stage_label']}</b></font>
        <font size=9 color='#64748B'>(Stage {patient_data['predicted_dr_stage']})</font><br/><br/>
        <font size=8 color='#64748B'>IMAGE MODEL CONFIDENCE</font><br/>
        <font size=11 color='#0A2342'><b>{patient_data['image_confidence']}%</b></font>"""
    else:
        stage_text = f"""<font size=13 color='#0A2342'><b>Clinical Risk Assessment</b></font><br/><br/>
        <font size=8 color='#64748B'>BASED ON</font><br/>
        <font size=10 color='#0A2342'>13 clinical parameters (no retinal image provided)</font>"""

    risk_card = Table([[gauge, Paragraph(stage_text, body_style)]], colWidths=[65*mm, 109*mm])
    risk_card.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,-1), risk_bg), ('BOX', (0,0), (-1,-1), 1, risk_border),
        ('VALIGN', (0,0), (-1,-1), 'MIDDLE'), ('TOPPADDING', (0,0), (-1,-1), 14), ('BOTTOMPADDING', (0,0), (-1,-1), 14),
        ('LEFTPADDING', (1,0), (1,0), 14),
    ]))
    elements.append(risk_card)
    elements.append(Spacer(1, 14))

    # ── Clinical parameters ──
    if has_clinical:
        elements.append(Paragraph("CLINICAL PARAMETERS", section_style))
        param_rows = [
            ('Age', f"{patient_data['age']} yrs", 'Diastolic BP', f"{patient_data['diastolic_bp']} mmHg"),
            ('Glucose', f"{patient_data['glucose']} mmol/L", 'BMI', f"{patient_data['bmi']} kg/m\u00b2"),
        ]
        clin_data = [[Paragraph(f"<font size=7 color='#64748B'><b>{a}</b></font>", body_style), Paragraph(f"<font size=10>{b}</font>", body_style),
                      Paragraph(f"<font size=7 color='#64748B'><b>{c}</b></font>", body_style), Paragraph(f"<font size=10>{d}</font>", body_style)]
                     for a,b,c,d in param_rows]
        clin_table = Table(clin_data, colWidths=[30*mm, 44*mm, 30*mm, 44*mm])
        clin_table.setStyle(TableStyle([
            ('GRID', (0,0), (-1,-1), 0.5, GRAY_LIGHT), ('BACKGROUND', (0,0), (0,-1), BG_LIGHT), ('BACKGROUND', (2,0), (2,-1), BG_LIGHT),
            ('TOPPADDING', (0,0), (-1,-1), 7), ('BOTTOMPADDING', (0,0), (-1,-1), 7), ('LEFTPADDING', (0,0), (-1,-1), 8),
        ]))
        elements.append(clin_table)
        elements.append(Spacer(1, 16))
    else:
        elements.append(Paragraph("CLINICAL PARAMETERS", section_style))
        elements.append(Paragraph("<i>No clinical data provided for this screening. Risk assessment based on retinal image analysis only.</i>", caption_style))
        elements.append(Spacer(1, 16))

    # ── Grad-CAM ──
    if has_image:
        elements.append(Paragraph("RETINAL IMAGE ANALYSIS \u2014 GRAD-CAM", section_style))
        img_card = Table([[RLImage(gradcam_path, width=60*mm, height=60*mm)]], colWidths=[174*mm])
        img_card.setStyle(TableStyle([('ALIGN', (0,0), (-1,-1), 'CENTER'), ('BOX', (0,0), (-1,-1), 0.5, GRAY_LIGHT), ('TOPPADDING',(0,0),(-1,-1),8), ('BOTTOMPADDING',(0,0),(-1,-1),8)]))
        elements.append(img_card)
        elements.append(Spacer(1, 6))
        elements.append(Paragraph(f"Heatmap highlights regions the model identified as most influential in classifying this retina as "
                                    f"<b>{patient_data['predicted_stage_label']}</b> (Stage {patient_data['predicted_dr_stage']}), "
                                    f"{patient_data['image_confidence']}% confidence.", caption_style))

    if has_clinical and has_image:
        elements.append(PageBreak())
    elif has_image and not has_clinical:
        elements.append(Spacer(1, 16))

    # ── SHAP ──
    if has_clinical:
        elements.append(Paragraph("CLINICAL RISK EXPLANATION \u2014 SHAP", section_style))
        shap_card = Table([[RLImage(shap_path, width=150*mm, height=95*mm)]], colWidths=[174*mm])
        shap_card.setStyle(TableStyle([('ALIGN', (0,0), (-1,-1), 'CENTER'), ('BOX', (0,0), (-1,-1), 0.5, GRAY_LIGHT), ('TOPPADDING',(0,0),(-1,-1),8), ('BOTTOMPADDING',(0,0),(-1,-1),8)]))
        elements.append(shap_card)
        elements.append(Spacer(1, 6))
        elements.append(Paragraph("Each bar shows how a clinical parameter shifted this patient's risk score, ranked by impact.", caption_style))
        elements.append(Spacer(1, 16))

    # ── Recommendations ──
    elements.append(Paragraph("CLINICAL RECOMMENDATIONS", section_style))
    if risk_score < 30:
        recs = ["Continue routine annual diabetic retinopathy screening.",
                "Maintain current glycaemic control \u2014 values are within a healthy range." if has_clinical else "Continue standard diabetic eye care schedule.",
                "No urgent ophthalmological referral indicated based on current screening result."]
        rec_color = GREEN
    elif risk_score < 60:
        recs = ["Schedule ophthalmological review within 3\u20136 months.",
                "Review glycaemic control and blood pressure management with primary physician." if has_clinical else "Recommend clinical data collection for a more complete risk assessment.",
                "Repeat screening in 6 months to monitor progression."]
        rec_color = AMBER
    else:
        recs = ["Refer to ophthalmologist for dilated fundus examination within 4 weeks.",
                "Urgent review of glycaemic control indicated." if has_clinical else "Recommend full clinical work-up to complement this image-based finding.",
                "Repeat screening in 3 months following specialist review."]
        rec_color = RED

    if not has_image:
        recs.insert(0, "No retinal image was provided \u2014 this assessment is based on clinical risk factors only. A retinal examination is recommended to confirm DR status.")
    if not has_clinical:
        recs.insert(0, "No clinical data was provided \u2014 this assessment is based on retinal imaging only. Clinical parameters are recommended for a more complete risk profile.")

    rec_rows = [[Paragraph(f"<font color='{rec_color.hexval()}'>\u25CF</font>", body_style), Paragraph(rec, body_style)] for rec in recs]
    rec_table = Table(rec_rows, colWidths=[6*mm, 168*mm])
    rec_table.setStyle(TableStyle([('VALIGN',(0,0),(-1,-1),'TOP'), ('TOPPADDING',(0,0),(-1,-1),3), ('BOTTOMPADDING',(0,0),(-1,-1),3)]))
    elements.append(rec_table)
    elements.append(Spacer(1, 18))

    # ── Disclaimer ──
    disclaimer_box = Table([[Paragraph(
        "<b>Disclaimer:</b> This report is generated by an AI screening system and must be reviewed and confirmed "
        "by a qualified clinician before any clinical decision is made. This system is a decision-support tool "
        "and is not a diagnostic device.", disclaimer_style)]], colWidths=[174*mm])
    disclaimer_box.setStyle(TableStyle([('BACKGROUND',(0,0),(-1,-1), BG_LIGHT), ('BOX',(0,0),(-1,-1),0.5,GRAY_LIGHT),
                                          ('TOPPADDING',(0,0),(-1,-1),8), ('BOTTOMPADDING',(0,0),(-1,-1),8), ('LEFTPADDING',(0,0),(-1,-1),8)]))
    elements.append(disclaimer_box)

    doc.build(elements)
    print(f"Report generated ({mode}): {output_path}")


# ── TEST ALL THREE MODES ──
import os
os.makedirs('/content/report_output', exist_ok=True)
for f in os.listdir('/content/report_output'):
    if f.endswith('.pdf'):
        os.remove(f'/content/report_output/{f}')

generate_dr_report_v3(
    patient_data=patient_data,
    shap_path='/content/report_output/patient_shap.png',
    gradcam_path=None,
    output_path='/content/report_output/DR_Report_Mode1_ClinicalOnly.pdf'
)

generate_dr_report_v3(
    patient_data=patient_data,
    gradcam_path='/content/report_output/patient_gradcam.png',
    shap_path=None,
    output_path='/content/report_output/DR_Report_Mode2_ImageOnly.pdf'
)

generate_dr_report_v3(
    patient_data=patient_data,
    gradcam_path='/content/report_output/patient_gradcam.png',
    shap_path='/content/report_output/patient_shap.png',
    output_path='/content/report_output/DR_Report_Mode3_Fusion.pdf'
)

Report generated (Mode 1 — Clinical Only): /content/report_output/DR_Report_Mode1_ClinicalOnly.pdf
Report generated (Mode 2 — Image Only): /content/report_output/DR_Report_Mode2_ImageOnly.pdf
Report generated (Mode 3 — Fusion): /content/report_output/DR_Report_Mode3_Fusion.pdf
